# Phase 6: Resolve Wikipedia references

Resolve each species in the Phase 5 coordinate release to an English Wikipedia page. The lookup follows Wikipedia title normalization and redirects, then records an association only when the exact title exists. Results are saved as a resumable table at `sites/pathogen-atlas-explorer/data/species_wikipedia_links.csv`; the web-data build consumes that table.

Re-running this notebook reuses completed lookup rows. Set `FORCE_REFRESH = True` only when you want to redo all Wikipedia lookups.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pathogen_atlas_v2").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Open this notebook from the Pathogen Atlas version2 repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent

COORDINATES_PATH = PROJECT_ROOT / "data/05_plots/release/species_feature_space_coordinates.parquet"
LOOKUP_TABLE_PATH = PROJECT_ROOT / "sites/pathogen-atlas-explorer/data/species_wikipedia_links.csv"
BUILDER_SCRIPT = PROJECT_ROOT / "sites/pathogen-atlas-explorer/scripts/build_wikipedia_links.py"
FORCE_REFRESH = False

if not COORDINATES_PATH.exists():
    raise FileNotFoundError(f"Run Phase 5 first; coordinates are missing: {COORDINATES_PATH}")

species_count = pd.read_parquet(COORDINATES_PATH, columns=["canonical_species"])["canonical_species"].nunique()
existing_count = 0
if LOOKUP_TABLE_PATH.exists():
    existing_count = len(pd.read_csv(LOOKUP_TABLE_PATH, keep_default_na=False))
display(Markdown(
    f"**Lookup scope:** {species_count:,} coordinate species  ·  "
    f"**Existing associations:** {existing_count:,} rows  ·  "
    f"**Force refresh:** {FORCE_REFRESH}"
))

**Lookup scope:** 1,703 coordinate species  ·  **Existing associations:** 1,703 rows  ·  **Force refresh:** False

## Resolve and save the Wikipedia association table

The resolver batches page-title queries, follows redirects, and writes progress to the CSV after each batch. This keeps completed mappings reusable if the notebook is interrupted.

In [2]:
command = [sys.executable, str(BUILDER_SCRIPT)]
if FORCE_REFRESH:
    command.append("--force-refresh")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

Resolving 0 species; reusing 1,703 existing rows.
Wrote sites/pathogen-atlas-explorer/data/species_wikipedia_links.csv: 1,703 species, 1,028 articles found.


CompletedProcess(args=['/Users/f.costa/.venvs/py312/bin/python', '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version2/sites/pathogen-atlas-explorer/scripts/build_wikipedia_links.py'], returncode=0)

In [3]:
wikipedia_links = pd.read_csv(LOOKUP_TABLE_PATH, keep_default_na=False)
wikipedia_found = wikipedia_links["match_status"].eq("found")
display(Markdown(
    f"**Saved:** `{LOOKUP_TABLE_PATH.relative_to(PROJECT_ROOT)}`  ·  "
    f"**Species:** {len(wikipedia_links):,}  ·  "
    f"**Wikipedia pages found:** {int(wikipedia_found.sum()):,}  ·  "
    f"**No exact article:** {int((~wikipedia_found).sum()):,}"
))
display(wikipedia_links.head(12))

**Saved:** `sites/pathogen-atlas-explorer/data/species_wikipedia_links.csv`  ·  **Species:** 1,703  ·  **Wikipedia pages found:** 1,028  ·  **No exact article:** 675

,canonical_species,wikipedia_title,wikipedia_url,match_status,match_type
0,Abiotrophia defectiva,Abiotrophia defectiva,https://en.wikipedia.org/wiki/Abiotrophia_defe...,found,exact_or_redirect
1,Acetobacter cibinongensis,,,not_found,exact_title_query
2,Acetobacter ghanensis,,,not_found,exact_title_query
3,Acetobacter indonesiensis,,,not_found,exact_title_query
4,Acetobacter tropicalis,,,not_found,exact_title_query
5,Achromobacter animicus,,,not_found,exact_title_query
6,Achromobacter denitrificans,Achromobacter denitrificans,https://en.wikipedia.org/wiki/Achromobacter_de...,found,exact_or_redirect
7,Achromobacter insolitus,Achromobacter insolitus,https://en.wikipedia.org/wiki/Achromobacter_in...,found,exact_or_redirect
8,Achromobacter marplatensis,Achromobacter marplatensis,https://en.wikipedia.org/wiki/Achromobacter_ma...,found,exact_or_redirect
9,Achromobacter mucicolens,,,not_found,exact_title_query
